# 06 GRPO：Group Rollout、相对奖励与 RL Readiness Audit

本节实现 GRPO 的核心机制，并把 RL Readiness Audit 放在训练入口之前。对同一 Prompt 采样一组回答，使用确定性 Verifier 产生 Outcome Reward，再计算组均值、组标准差、Group Advantage、Policy Ratio 和 PPO 风格 Clipped Objective。

GRPO 不保证提升所有任务。若一个 Group 全对或全错，Binary Reward 的组内方差为零，中心化 Advantage 没有有效区分信号；Notebook 必须记录这一失败模式，而不是人为篡改 Reward 制造 Mixed Group。

DGX Spark 阶段不加载模型、不启动 vLLM、不运行真实 Rollout 或 GPU 更新。所有 CPU Fixture 都带有 `fixture=True` 标记，仅用于数值和审计测试。

## 学习目标

- 对 Equation Numeric 和 Bit Manipulation 进行多次候选 Rollout；
- 统计 All-correct、Mixed、All-wrong、Reward Variance 和 Zero-Variance Ratio；
- 推导并实现 `A_i=(r_i-mean(r))/(std(r)+epsilon)`；
- 对 Completion Token 应用 Importance Ratio、Clipping 和 Policy Logprob Change 诊断；
- 根据 Readiness Audit 决定是否进入 GRPO，而不是默认扩大训练步数。

## 1. 任务定位与 Readiness Audit

Gravity 主要用于 Baseline、Prompt 和 SFT 的能力保持，通常不进入 GRPO 主训练。Equation Numeric 和 Bit Manipulation 是主要 RL 任务。首轮 Group Size `G=8`，但 Temperature 和 G 必须在 RTX 5090 上校准。

对每道题：

- **All-correct**：`N/N` 正确，主要用于能力保持；
- **Mixed**：`1..N-1` 正确，GRPO 优先训练池；
- **All-wrong**：`0/N` 正确，检查探索、SFT 覆盖和 Reward；
- **Zero-variance**：奖励标准差为零，组内相对更新信号退化。

理论上若单次正确率为 `p`，独立采样 `N` 次：`P(Mixed)=1-p^N-(1-p)^N`。这只是近似，真实 Rollout 可能相关，不能用它伪造实验结果。

In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
RESULT_DIR = PROJECT_ROOT / "data" / "processed" / "grpo"
CHECKPOINT_ROOT = PROJECT_ROOT / "data" / "processed" / "checkpoints"
SFT_MANIFEST_PATH = CHECKPOINT_ROOT / "sft_manifest.json"
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "train.csv"
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
BASE_MODEL_PATH = os.environ.get("QWEN3_MODEL_PATH", MODEL_ID)
RESULT_DIR = RESULT_DIR / os.environ.get("GPU_SMOKE_RUN_ID", "")
CHECKPOINT_ROOT = CHECKPOINT_ROOT / os.environ.get("GPU_SMOKE_RUN_ID", "")
SFT_MANIFEST_PATH = CHECKPOINT_ROOT / "sft_manifest.json"
SEED = 42
RUN_GRPO = os.environ.get("RUN_GRPO", "0") == "1"  # umbrella flag
RUN_GRPO_ROLLOUT = os.environ.get("RUN_GRPO_ROLLOUT", "1" if RUN_GRPO else "0") == "1"
RUN_GRPO_TRAINING = os.environ.get("RUN_GRPO_TRAINING", "1" if RUN_GRPO else "0") == "1"
RUN_MODE = os.environ.get("RUN_MODE", "smoke")
GROUP_SIZE = 8
SMOKE_PROMPTS_PER_TASK = 2
MAX_NEW_TOKENS = int(os.environ.get("RL_MAX_NEW_TOKENS", "2048"))
MAX_MODEL_LEN = int(os.environ.get("RL_MAX_MODEL_LEN", "8192"))
TEMPERATURE = 1.0
TOP_P = 1.0
TRAINING_PROFILE = "official-pt-rl-cuda"
ADVANTAGE_EPS = 1e-6
GRPO_CLIP_EPS = 0.2
KL_COEF = 0.0
LEARNING_RATE = 1e-5
GRPO_MICRO_BATCH_SIZE = int(os.environ.get("GRPO_MICRO_BATCH_SIZE", "1"))
GRPO_GRAD_ACCUMULATION_STEPS = int(os.environ.get("GRPO_GRAD_ACCUMULATION_STEPS", "32"))
GRPO_ROLLOUT_PATH = RESULT_DIR / "grpo_smoke_rollouts.jsonl"
GRPO_ROLLOUT_MANIFEST_PATH = RESULT_DIR / "grpo_rollout_manifest.json"
GRPO_TRAINING_MANIFEST_PATH = RESULT_DIR / "grpo_training_manifest.json"
TARGET_TASKS = ("equation_numeric", "bit_manipulation")

np.random.seed(SEED)
torch.manual_seed(SEED)
print({"model_id": MODEL_ID, "run_grpo": RUN_GRPO, "run_mode": RUN_MODE, "group_size": GROUP_SIZE, "tasks": TARGET_TASKS})

## 2. Group Rollout Schema 与确定性 Reward

每条回答保留 Prompt ID、任务、Completion Token IDs、Response Mask、策略版本、采样参数、Verifier 版本和最终 Reward。真实运行必须保存原始输出和截断状态；只保存 Group 平均分不足以审计零方差问题。

In [ ]:
@dataclass
class GroupMember:
    prompt_id: str
    task_type: str
    prompt: str
    completion: str
    completion_token_ids: list[int]
    response_mask: list[int]
    reward: float
    correct: bool
    finish_reason: str
    truncated: bool
    policy_version: str
    verifier_version: str
    temperature: float
    top_p: float
    source: str
    fixture: bool = False


def group_member_to_dict(member: GroupMember) -> dict[str, object]:
    return {
        "prompt_id": member.prompt_id, "task_type": member.task_type, "prompt": member.prompt, "expected_answer": getattr(member, "expected_answer", ""),
        "completion": member.completion, "completion_token_ids": member.completion_token_ids,
        # CPU fixtures include the same alignment metadata as real vLLM records.
        "prompt_token_ids": [101, 102, 103],
        "behavior_logprobs": [-0.2] * len(member.completion_token_ids),
        "response_mask": member.response_mask, "reward": member.reward, "correct": member.correct,
        "finish_reason": member.finish_reason, "truncated": member.truncated,
        "policy_version": member.policy_version, "verifier_version": member.verifier_version,
        "temperature": member.temperature, "top_p": member.top_p, "source": member.source,
        "fixture": member.fixture,
    }


def deterministic_binary_reward(correct: bool) -> float:
    return 1.0 if bool(correct) else 0.0


def build_cpu_group_fixture(group_size: int = GROUP_SIZE) -> list[dict[str, object]]:
    # Algebra fixture: groups deliberately cover all categories for audit tests;
    # these labels are not model outputs and cannot be used as experiment scores.
    patterns = {
        "group_all_correct": [1] * group_size,
        "group_mixed": [1, 0] * (group_size // 2),
        "group_all_wrong": [0] * group_size,
    }
    members = []
    for name, rewards in patterns.items():
        for index, reward in enumerate(rewards):
            members.append(group_member_to_dict(GroupMember(
                prompt_id=name, task_type="bit_manipulation", prompt="CPU fixture", completion=f"candidate-{index}",
                completion_token_ids=[1, 2, index + 3], response_mask=[0, 0, 0, 1, 1, 1], reward=float(reward), correct=bool(reward),
                finish_reason="stop", truncated=False, policy_version="cpu_fixture", verifier_version="toy_verifier",
                temperature=TEMPERATURE, top_p=TOP_P, source="cpu_fixture_only", fixture=True,
            )))
    return members

## 3. RL Readiness Audit

Audit 不能人为把全错答案改成错误或把全对答案改成正确。它只统计实际 Verifier Reward 的分布，并明确标记 `fixture_only`。Mixed Group 比例过低时，应优先检查 Temperature、SFT 覆盖、轨迹质量和可验证 Process Reward。

In [ ]:
def readiness_audit(group_members: list[dict[str, object]], group_size: int = GROUP_SIZE) -> dict[str, object]:
    grouped = defaultdict(list)
    for member in group_members:
        grouped[(member["prompt_id"], member["task_type"])].append(member)
    categories = Counter()
    variances = []
    reward_counts = []
    for key, members in grouped.items():
        if len(members) != group_size:
            categories["invalid_group_size"] += 1
            continue
        rewards = np.asarray([float(member["reward"]) for member in members], dtype=np.float32)
        correct_count = int(np.sum(rewards > 0.5))
        reward_counts.append(correct_count)
        variances.append(float(np.var(rewards)))
        if correct_count == group_size:
            categories["all_correct"] += 1
        elif correct_count == 0:
            categories["all_wrong"] += 1
        else:
            categories["mixed"] += 1
    valid_groups = categories["all_correct"] + categories["mixed"] + categories["all_wrong"]
    return {
        "groups": valid_groups,
        "invalid_groups": categories["invalid_group_size"],
        "categories": dict(categories),
        "all_correct_ratio": categories["all_correct"] / valid_groups if valid_groups else 0.0,
        "mixed_ratio": categories["mixed"] / valid_groups if valid_groups else 0.0,
        "all_wrong_ratio": categories["all_wrong"] / valid_groups if valid_groups else 0.0,
        "mean_reward_variance": float(np.mean(variances)) if variances else 0.0,
        "zero_variance_ratio": float(np.mean(np.asarray(variances) <= 1e-12)) if variances else 0.0,
        "correct_count_histogram": dict(Counter(reward_counts)),
        "fixture_only": bool(group_members) and all(member.get("fixture", False) for member in group_members),
    }


def theoretical_mixed_probability(p: float, group_size: int = GROUP_SIZE) -> float:
    p = float(p)
    return 1.0 - p ** group_size - (1.0 - p) ** group_size


cpu_fixture_members = build_cpu_group_fixture()
print(json.dumps({"readiness_audit_fixture": readiness_audit(cpu_fixture_members), "theoretical_mixed_at_p_0_5": theoretical_mixed_probability(0.5)}, ensure_ascii=False, indent=2))

## 4. Group Advantage、Ratio、Clipping 和 Policy Logprob Change

Group Advantage 使用组内 Population Standard Deviation（`ddof=0`）。零方差组的 Advantage 设为零并记录，不添加人工噪声。策略更新只看 Response Token；Prompt 和 Padding 的 Mask 为零。

In [ ]:
def group_advantage(rewards: torch.Tensor, eps: float = ADVANTAGE_EPS) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """rewards: [batch, group]. Return advantage, mean and population std."""
    if rewards.ndim != 2:
        raise ValueError("rewards must be [batch, group]")
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    advantages = (rewards - mean) / (std + eps)
    zero_variance = std.squeeze(1) <= eps
    advantages = torch.where(zero_variance.unsqueeze(1), torch.zeros_like(advantages), advantages)
    return advantages, mean.squeeze(1), std.squeeze(1)


def grpo_clip_loss(old_logprobs: torch.Tensor, new_logprobs: torch.Tensor, advantages: torch.Tensor, response_mask: torch.Tensor, clip_eps: float = GRPO_CLIP_EPS) -> dict[str, torch.Tensor]:
    """Token-level objective with one scalar group advantage per completion."""
    if old_logprobs.shape != new_logprobs.shape or old_logprobs.shape != response_mask.shape:
        raise ValueError("logprobs and response_mask must align")
    ratio = torch.exp((new_logprobs - old_logprobs).clamp(-20, 20))
    expanded_advantage = advantages.unsqueeze(-1).expand_as(ratio)
    unclipped = ratio * expanded_advantage
    clipped_ratio = ratio.clamp(1.0 - clip_eps, 1.0 + clip_eps)
    clipped = clipped_ratio * expanded_advantage
    objective = torch.minimum(unclipped, clipped)
    clip_fraction = ((ratio - clipped_ratio).abs() > 1e-8).to(torch.float32)
    mask = response_mask.to(torch.float32)
    denom = mask.sum().clamp_min(1.0)
    return {
        "loss": -(objective * mask).sum() / denom,
        "ratio": ratio.detach(),
        "clip_fraction": (clip_fraction * mask).sum() / denom,
    }


def policy_logprob_change(old_logprobs: torch.Tensor, new_logprobs: torch.Tensor, response_mask: torch.Tensor) -> torch.Tensor:
    mask = response_mask.to(torch.float32)
    return ((old_logprobs - new_logprobs) * mask).sum() / mask.sum().clamp_min(1.0)


def reward_process_bonus(example_verified: torch.Tensor, query_verified: torch.Tensor, alpha: float = 0.1) -> torch.Tensor:
    """Optional deterministic process reward; final answer remains dominant."""
    return query_verified.to(torch.float32) + alpha * example_verified.to(torch.float32)

## 5. Tiny Policy 与一次 GRPO Update

Tiny Policy 只用于验证梯度和张量形状。真实模型更新需要最佳 SFT Checkpoint、真实 Rollout Logprobs 和独立 Verifier Reward。

In [ ]:
class TinyPolicy(nn.Module):
    def __init__(self, vocab_size: int = 32, hidden_size: int = 20):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)
    def forward(self, input_ids, attention_mask=None, use_cache=None):
        return self.head(self.embedding(input_ids))


def token_logprobs(logits: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
    return F.log_softmax(logits[:, :-1, :].float(), dim=-1).gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)


def one_grpo_update(policy: nn.Module, optimizer: torch.optim.Optimizer, input_ids: torch.Tensor, response_mask: torch.Tensor, old_logprobs: torch.Tensor, advantages: torch.Tensor, kl_coef: float = KL_COEF) -> dict[str, float]:
    logits = policy(input_ids)
    new_logprobs = token_logprobs(logits, input_ids)
    aligned_mask = response_mask[:, 1:]
    objective = grpo_clip_loss(old_logprobs, new_logprobs, advantages, aligned_mask)
    kl = policy_logprob_change(old_logprobs, new_logprobs, aligned_mask)
    total = objective["loss"] + kl_coef * kl
    optimizer.zero_grad(set_to_none=True)
    total.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer.step()
    return {"loss": float(total.detach()), "policy_loss": float(objective["loss"].detach()), "policy_logprob_change": float(kl.detach()), "clip_fraction": float(objective["clip_fraction"].detach())}

## 6. vLLM Group Rollout / GRPO 入口（当前关闭）

真实运行时每个 Prompt 生成 `GROUP_SIZE` 条候选，并把 vLLM 输出 Token IDs、Logprobs、finish_reason 和策略版本保存下来；然后关闭 vLLM，切换训练环境执行一轮 GRPO Update。不能用人为构造的 Reward Variance 代替真实 Readiness Audit。

In [ ]:
def run_vllm_group_rollout(prompt_rows, model_source, policy_version, adapter_path=None):
    records = _generate_rollout_records(prompt_rows, model_source or BASE_MODEL_PATH, policy_version, adapter_path, GROUP_SIZE)
    return attach_verifier_rewards(records, lambda r: independent_answer_verifier(r["task_type"], r["completion"], r["expected_answer"], r["truncated"]))


## 7. CPU Smoke Test：零方差、Group Advantage 与 Clipping

测试覆盖三种 Group 类别、零方差 Advantage、混合组正负优势、Ratio/Clip、Policy Logprob Change 和反向传播。Fixture 只验证公式；真实 Group 分布必须由明天的模型 Rollout 得到。

In [ ]:
audit = readiness_audit(cpu_fixture_members)
assert audit["groups"] == 3
assert audit["categories"]["all_correct"] == 1
assert audit["categories"]["mixed"] == 1
assert audit["categories"]["all_wrong"] == 1
assert audit["zero_variance_ratio"] == 2 / 3

rewards = torch.tensor([[1.0] * GROUP_SIZE, [1.0, 0.0] * (GROUP_SIZE // 2), [0.0] * GROUP_SIZE])
advantages, means, stds = group_advantage(rewards)
assert torch.allclose(advantages[0], torch.zeros(GROUP_SIZE))
assert torch.allclose(advantages[2], torch.zeros(GROUP_SIZE))
assert torch.isfinite(advantages[1]).all()
assert torch.isclose(stds[1], torch.tensor(0.5))

batch_size, time_steps, vocab_size = 3, 7, 32
torch.manual_seed(SEED)
# Flatten one Group's members into the rollout batch; each member receives its
# scalar Group Advantage while sharing the same Prompt group statistics.
input_ids = torch.randint(0, vocab_size, (batch_size * GROUP_SIZE, time_steps))
response_mask = torch.zeros(batch_size * GROUP_SIZE, time_steps, dtype=torch.float32)
response_mask[:, 2:] = 1
old_logprobs = torch.randn(batch_size * GROUP_SIZE, time_steps - 1) * 0.1
member_advantages = advantages.reshape(-1)
policy = TinyPolicy(vocab_size=vocab_size)
optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-2)
logs = one_grpo_update(policy, optimizer, input_ids, response_mask, old_logprobs, member_advantages)
assert all(math.isfinite(value) for value in logs.values())
assert any(parameter.grad is not None for parameter in policy.parameters())

# Explicitly check that binary all-correct/all-wrong groups have no update advantage.
assert float(advantages[[0, 2]].abs().sum()) == 0.0

grpo_cpu_report = {
    "status": "passed",
    "readiness_audit": audit,
    "zero_variance_advantage_zero": True,
    "mixed_group_advantage_finite": True,
    "grpo_loss_finite": True,
    "policy_gradient": True,
    "gpu_rollout": "not_run",
    "gpu_training": "not_run",
}
print(json.dumps(grpo_cpu_report, ensure_ascii=False, indent=2))

## 本节总结与 RL 决策

GRPO 的首要产物不是一条更高的分数，而是可审计的 Readiness Audit。如果真实 Rollout 主要是 All-correct 或 All-wrong，必须记录 Zero-Variance Ratio，检查采样探索、SFT 覆盖和确定性过程奖励；不能通过改标签制造 Mixed Group。只有 Mixed Group 足够且 Reward 与独立答案指标一致时，才进入 G1 Binary Outcome GRPO。

当前未运行真实 GPU Rollout、模型更新、显存、吞吐和 SFT→GRPO 对比，均标记为 `Not Run`。DPO、PPO、GRPO 应从同一个最佳 SFT Checkpoint 平行开始，不能串联训练后再比较。

## 8. 独立 Verifier、Rollout 对齐与输入审计

真实 GRPO 的奖励必须在 Rollout 之后由独立 Verifier 计算。Generator、模型自评和
Reward 计算不共享答案函数。下面的适配器只负责把已有任务 Verifier 的结果写入
Rollout 记录；如果记录不完整、行为策略 Logprob 缺失或 Group 大小不正确，训练
入口直接拒绝。数值比较使用项目统一的 `math.isclose` 标准，Bit 结果使用 8-bit
精确匹配。示例中的通用提取器只用于接口 Smoke Test；正式实验应传入第一个
Notebook 生成的任务专用 Verifier。


### 8.1 Rollout Schema 与单条输入审计

In [ ]:
import copy
import re
from typing import Any, Callable, Mapping, Sequence

REQUIRED_ROLLOUT_FIELDS = {
    "prompt_id", "task_type", "prompt", "expected_answer", "completion", "completion_token_ids",
    "behavior_logprobs", "finish_reason", "truncated", "policy_version",
}


def validate_rollout_record(record: Mapping[str, object]) -> list[str]:
    errors = [f"missing:{key}" for key in sorted(REQUIRED_ROLLOUT_FIELDS) if key not in record]
    if errors:
        return errors
    token_ids = record["completion_token_ids"]
    behavior_logprobs = record["behavior_logprobs"]
    if not isinstance(token_ids, Sequence) or isinstance(token_ids, (str, bytes)):
        errors.append("completion_token_ids_not_sequence")
    if not isinstance(behavior_logprobs, Sequence) or isinstance(behavior_logprobs, (str, bytes)):
        errors.append("behavior_logprobs_not_sequence")
    if isinstance(token_ids, Sequence) and isinstance(behavior_logprobs, Sequence) and len(token_ids) != len(behavior_logprobs):
        errors.append("token_logprob_length_mismatch")
    if isinstance(behavior_logprobs, Sequence) and not isinstance(behavior_logprobs, (str, bytes)):
        try:
            if any(not math.isfinite(float(value)) for value in behavior_logprobs):
                errors.append("non_finite_behavior_logprob")
        except (TypeError, ValueError):
            errors.append("behavior_logprobs_not_numeric")
    if "response_mask" in record:
        prompt_ids = record.get("prompt_token_ids", [])
        expected_mask_length = len(prompt_ids) + len(token_ids)
        if len(record["response_mask"]) != expected_mask_length:
            errors.append("response_mask_length_mismatch")
        elif any(int(value) != 0 for value in record["response_mask"][:len(prompt_ids)]):
            errors.append("prompt_response_mask_not_zero")
        elif any(int(value) != 1 for value in record["response_mask"][len(prompt_ids):]):
            errors.append("completion_response_mask_not_one")
    return errors


### 8.2 Group 完整性与任务专用 Verifier

In [ ]:
def validate_group_members(group_members: Sequence[Mapping[str, object]], group_size: int = GROUP_SIZE) -> dict[str, object]:
    errors = []
    for index, member in enumerate(group_members):
        member_errors = validate_rollout_record(member)
        if "reward" not in member or "correct" not in member:
            member_errors.append("missing:verifier_reward")
        else:
            reward = float(member["reward"])
            if not math.isfinite(reward) or reward < 0.0 or reward > 1.0:
                member_errors.append("reward_out_of_range")
        if member_errors:
            errors.append({"index": index, "prompt_id": member.get("prompt_id"), "errors": member_errors})
    groups = Counter((member.get("prompt_id"), member.get("task_type")) for member in group_members)
    bad_group_sizes = {str(key): count for key, count in groups.items() if count != group_size}
    return {
        "valid": not errors and not bad_group_sizes,
        "record_errors": errors,
        "bad_group_sizes": bad_group_sizes,
        "records": len(group_members),
        "groups": len(groups),
    }


OFFICIAL_VERIFIER_VERSION = "official_nvidia_metric_v3"


def _normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    return re.sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=re.I).strip()


def _extract_final_answer(text: str, task_type: str) -> tuple[str | None, str]:
    """Extract an explicitly marked final answer; never use an arbitrary CoT number."""
    import re
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return _normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?P<tag>final|answer)>\s*(?P<value>.*?)\s*</(?P=tag)>", text, flags=re.I | re.S)
    if tagged:
        return _normalize_answer_text(tagged[-1][1]), "tagged"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        labeled = re.match(r"(?i)^(?:final answer|answer|最终答案|最终输出)\s*[:：]\s*(.+)$", line)
        if labeled:
            candidate = _normalize_answer_text(labeled.group(1)).strip("`$")
            if task_type == "bit_manipulation" and re.fullmatch(r"(?:0b)?[01]{8}", candidate):
                return candidate.removeprefix("0b"), "final_label"
            if task_type in {"gravity", "equation_numeric"} and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", candidate):
                return candidate, "final_label"
        if task_type == "bit_manipulation" and re.fullmatch(r"(?:0b)?[01]{8}", line):
            return line.removeprefix("0b"), "standalone_final_line"
        if task_type in {"gravity", "equation_numeric"} and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", line):
            return line, "standalone_final_line"
    return None, "unmarked_or_empty"


def _verify_answer(task_type: str, predicted: object, expected: object) -> tuple[bool, str]:
    if predicted is None:
        return False, "not_extracted"
    predicted_text, expected_text = _normalize_answer_text(predicted), _normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p, e = predicted_text.removeprefix("0b"), expected_text.removeprefix("0b")
        if not re.fullmatch(r"[01]{8}", p) or not re.fullmatch(r"[01]{8}", e):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        if not re.fullmatch(r"[-+]?\d+(?:\.\d+)?", predicted_text):
            return False, "invalid_numeric_format"
        try:
            ok = math.isclose(float(predicted_text), float(expected_text), rel_tol=1e-2, abs_tol=1e-5)
        except (TypeError, ValueError):
            ok = False
        return ok, "official_math_isclose"
    return predicted_text == expected_text, "exact_text"


def independent_answer_verifier(task_type: str, completion: str, expected_answer: str, truncated: bool = False) -> dict[str, object]:
    """Independent answer-only adapter using the Notebook 01 verifier contract."""
    if truncated:
        return {"correct": False, "reward": 0.0, "predicted": None, "verification_method": "truncated_output", "verifier_version": OFFICIAL_VERIFIER_VERSION}
    predicted, extraction_method = _extract_final_answer(completion, task_type)
    correct, verification_method = _verify_answer(task_type, predicted, expected_answer)
    return {
        "correct": correct,
        "reward": 1.0 if correct else 0.0,
        "predicted": predicted,
        "extraction_method": extraction_method,
        "verification_method": verification_method,
        "verifier_version": OFFICIAL_VERIFIER_VERSION,
    }


### 8.3 附加 Verifier Reward 与 Readiness Audit

In [ ]:
def attach_verifier_rewards(
    rollout_records: Sequence[Mapping[str, object]],
    verifier: Callable[[Mapping[str, object]], Mapping[str, object]],
) -> list[dict[str, object]]:
    """Attach deterministic Verifier output without changing the Prompt."""
    enriched = []
    for record in rollout_records:
        errors = validate_rollout_record(record)
        if errors:
            raise ValueError(f"invalid rollout record {record.get('prompt_id')}: {errors}")
        outcome = dict(verifier(record))
        if "correct" not in outcome or "reward" not in outcome:
            raise ValueError("verifier must return correct and reward")
        enriched_record = dict(record)
        enriched_record.update({
            "correct": bool(outcome["correct"]),
            "reward": float(outcome["reward"]),
            "verifier_version": str(outcome.get("verifier_version", "unknown")),
            "verification_details": outcome,
        })
        enriched.append(enriched_record)
    return enriched


def audit_group_members(group_members: Sequence[Mapping[str, object]], group_size: int = GROUP_SIZE) -> dict[str, object]:
    validation = validate_group_members(group_members, group_size)
    if not validation["valid"]:
        return {"status": "invalid_input", "validation": validation, "fixture_only": False}
    audit = readiness_audit([dict(member) for member in group_members], group_size)
    audit["status"] = "ok"
    audit["validation"] = validation
    audit["reward_definition"] = "binary_outcome_reward"
    return audit


### 8.4 Group Reward Tensor 与 Token 对齐 Batch

In [ ]:
def build_group_reward_tensors(group_members: Sequence[Mapping[str, object]], group_size: int = GROUP_SIZE) -> dict[str, object]:
    validation = validate_group_members(group_members, group_size)
    if not validation["valid"]:
        raise ValueError(json.dumps(validation, ensure_ascii=False))
    grouped = defaultdict(list)
    for member in group_members:
        grouped[(str(member["prompt_id"]), str(member["task_type"]))].append(member)
    keys = sorted(grouped)
    rewards = torch.tensor([[float(member["reward"]) for member in grouped[key]] for key in keys], dtype=torch.float32)
    advantages, means, stds = group_advantage(rewards)
    return {"group_keys": keys, "rewards": rewards, "advantages": advantages, "mean_rewards": means, "std_rewards": stds}


def prepare_grpo_batch(group_members: Sequence[Mapping[str, object]], pad_token_id: int = 0, device: str = "cpu") -> dict[str, torch.Tensor]:
    """Pad full Prompt+Completion sequences and align behavior Logprobs.

    `old_logprobs[:, k-1]` corresponds to the token at full-sequence position
    k. The first completion token therefore starts at `prompt_length - 1`.
    Prompt tokens have response mask zero and never contribute to the objective.
    """
    individual_errors = []
    for index, member in enumerate(group_members):
        member_errors = validate_rollout_record(member)
        if "reward" not in member or "correct" not in member:
            member_errors.append("missing:verifier_reward")
        if member_errors:
            individual_errors.append({"index": index, "errors": member_errors})
    if individual_errors:
        raise ValueError(json.dumps({"valid": False, "record_errors": individual_errors}, ensure_ascii=False))
    sequences, prompt_lengths, completion_lengths, old_values = [], [], [], []
    for member in group_members:
        prompt_ids = list(member.get("prompt_token_ids", []))
        completion_ids = list(member["completion_token_ids"])
        if not prompt_ids:
            raise ValueError("每条真实 Rollout 必须保留 prompt_token_ids")
        sequences.append(prompt_ids + completion_ids)
        prompt_lengths.append(len(prompt_ids))
        completion_lengths.append(len(completion_ids))
        old_values.append([float(value) for value in member["behavior_logprobs"]])
    max_len = max(len(sequence) for sequence in sequences)
    batch = len(sequences)
    input_ids = torch.full((batch, max_len), int(pad_token_id), dtype=torch.long, device=device)
    attention_mask = torch.zeros((batch, max_len), dtype=torch.long, device=device)
    response_mask = torch.zeros((batch, max_len), dtype=torch.float32, device=device)
    old_logprobs = torch.zeros((batch, max_len - 1), dtype=torch.float32, device=device)
    for row, (sequence, prompt_len, completion_len, values) in enumerate(zip(sequences, prompt_lengths, completion_lengths, old_values)):
        length = len(sequence)
        input_ids[row, :length] = torch.tensor(sequence, dtype=torch.long, device=device)
        attention_mask[row, :length] = 1
        response_mask[row, prompt_len:length] = 1
        if values:
            start = prompt_len - 1
            old_logprobs[row, start:start + completion_len] = torch.tensor(values, dtype=torch.float32, device=device)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "response_mask": response_mask, "old_logprobs": old_logprobs}


## 9. CausalLM GRPO Update 入口

下面的更新函数与 Tiny Policy 使用同一套 Group Advantage、Response Mask、Ratio、Clipping 和 Logprob Change；它读取 Transformers CausalLM 的 `outputs.logits`，不使用高层 GRPO Trainer。真实运行前必须通过 Readiness Audit；没有 Mixed Group 时直接返回阻塞状态。

In [ ]:
def _slice_grpo_batch(batch: Mapping[str, torch.Tensor], start: int, end: int) -> dict[str, torch.Tensor]:
    return {key: value[start:end] for key, value in batch.items()}


In [ ]:
def causal_lm_grpo_update(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: Mapping[str, torch.Tensor] | Sequence[Mapping[str, torch.Tensor]],
    member_advantages: torch.Tensor | Sequence[torch.Tensor],
    kl_coef: float = KL_COEF,
    micro_batch_size: int = GRPO_MICRO_BATCH_SIZE,
    gradient_accumulation_steps: int = GRPO_GRAD_ACCUMULATION_STEPS,
) -> dict[str, float]:
    """Update GRPO with micro-batches while keeping G=8 advantages intact.

    Group rewards and advantages are computed before this function. A long
    rollout group is then split only for the forward/backward pass, so 32
    responses do not need to coexist in one activation graph.
    """
    if micro_batch_size < 1 or gradient_accumulation_steps < 1:
        raise ValueError("micro_batch_size and gradient_accumulation_steps must be >= 1")
    if isinstance(batch, Mapping):
        full_size = int(batch["input_ids"].shape[0])
        batches = [_slice_grpo_batch(batch, start, min(start + micro_batch_size, full_size)) for start in range(0, full_size, micro_batch_size)]
        advantages = [member_advantages[start:min(start + micro_batch_size, full_size)] for start in range(0, full_size, micro_batch_size)]
    else:
        batches = list(batch)
        advantages = list(member_advantages)
    if len(batches) != len(advantages) or not batches:
        raise ValueError("GRPO micro-batch and advantage counts do not align")
    model.train()
    for module in model.modules():
        if isinstance(module, nn.Dropout): module.eval()
    optimizer.zero_grad(set_to_none=True)
    total_tokens = 0
    total_loss = total_policy = total_change = total_clip = 0.0
    optimizer_steps = 0
    for window_start in range(0, len(batches), gradient_accumulation_steps):
        window_batches = batches[window_start:window_start + gradient_accumulation_steps]
        window_advs = advantages[window_start:window_start + gradient_accumulation_steps]
        token_counts = [int(batch_item["response_mask"][:, 1:].sum().item()) for batch_item in window_batches]
        window_tokens = sum(token_counts)
        if window_tokens == 0:
            optimizer.zero_grad(set_to_none=True)
            continue
        for batch_item, advantage_item, token_count in zip(window_batches, window_advs, token_counts):
            outputs = model(input_ids=batch_item["input_ids"], attention_mask=batch_item.get("attention_mask"), use_cache=False)
            logits = outputs if isinstance(outputs, torch.Tensor) else (outputs.logits if hasattr(outputs, "logits") else outputs["logits"])
            new_logprobs = token_logprobs(logits, batch_item["input_ids"])
            aligned_mask = batch_item["response_mask"][:, 1:]
            old_logprobs = batch_item["old_logprobs"]
            objective = grpo_clip_loss(old_logprobs, new_logprobs, torch.as_tensor(advantage_item, device=new_logprobs.device), aligned_mask)
            change = policy_logprob_change(old_logprobs, new_logprobs, aligned_mask)
            scale = token_count / max(window_tokens, 1)
            (objective["loss"] + kl_coef * change).mul(scale).backward()
            total_loss += float(objective["loss"].detach()) * scale
            total_policy += float(objective["loss"].detach()) * scale
            total_change += float(change.detach()) * scale
            total_clip += float(objective["clip_fraction"].detach()) * scale
            total_tokens += token_count
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        optimizer_steps += 1
    if total_tokens == 0:
        return {"status": "blocked_no_response_tokens", "optimizer_steps": 0.0, "response_tokens": 0.0}
    return {"status": "completed", "loss": total_loss / max(optimizer_steps, 1), "policy_loss": total_policy / max(optimizer_steps, 1), "policy_logprob_change": total_change / max(optimizer_steps, 1), "clip_fraction": total_clip / max(optimizer_steps, 1), "response_tokens": float(total_tokens), "optimizer_steps": float(optimizer_steps), "micro_batches": float(len(batches)), "micro_batch_size": float(micro_batch_size), "gradient_accumulation_steps": float(gradient_accumulation_steps)}


### 9.1 读取 SFT Manifest 与加载训练 Adapter

In [ ]:
def read_sft_manifest(path: Path = SFT_MANIFEST_PATH) -> dict[str, object]:
    if not path.exists(): raise FileNotFoundError(f"SFT Manifest 不存在：{path}；请先运行 Notebook 03")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != "nemotron-sft-checkpoint/v1" or manifest.get("base_model", {}).get("id") != MODEL_ID:
        raise ValueError("SFT Manifest schema 或 Base Model 不匹配")
    if manifest.get("status") != "completed":
        raise ValueError("SFT Manifest 尚未标记为 completed，不能初始化 GRPO")
    adapter_path = Path(resolve_manifest_path(str(manifest.get("adapter_path", ""))))
    if not adapter_path.is_dir() or not (adapter_path / "adapter_config.json").exists():
        raise FileNotFoundError(f"SFT Adapter 不存在或缺少 adapter_config.json：{adapter_path}")
    return manifest


def resolve_manifest_path(value: str) -> str:
    path = Path(value)
    return str(path if path.is_absolute() else PROJECT_ROOT / path)


def _load_sft_policy(manifest: dict[str, object], mode: str):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel, prepare_model_for_kbit_training
    kwargs = {"trust_remote_code": True, "device_map": {"":0}, "attn_implementation":"sdpa", "torch_dtype":torch.bfloat16}
    if mode == "nf4_qlora": kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    elif mode == "bf16_lora": kwargs["torch_dtype"] = torch.bfloat16
    else: raise ValueError(mode)
    base = AutoModelForCausalLM.from_pretrained(manifest["base_model"]["path"], **kwargs)
    if mode == "nf4_qlora": base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    model = PeftModel.from_pretrained(base, resolve_manifest_path(str(manifest["adapter_path"])), is_trainable=True)
    tokenizer = AutoTokenizer.from_pretrained(manifest.get("tokenizer_path", manifest["base_model"]["path"]), trust_remote_code=True)
    model.config.use_cache = False
    model.gradient_checkpointing_enable(); model.enable_input_require_grads(); model.eval()
    return model, tokenizer


### 9.2 保存 GRPO Adapter 与训练 Manifest

In [ ]:
def save_grpo_adapter(model, tokenizer, parent_manifest_path: Path, mode: str, audit: dict, logs: dict[str, float], parent_manifest: dict[str, object] | None = None) -> dict[str, object]:
    output_dir = CHECKPOINT_ROOT / f"grpo_{mode}"
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    parent_manifest = parent_manifest or read_sft_manifest(parent_manifest_path)
    manifest = {"schema_version":"nemotron-grpo-checkpoint/v1", "status":"pending_reload", "parent_sft_manifest":str(parent_manifest_path.relative_to(PROJECT_ROOT)), "parent_sft_adapter":str(parent_manifest["adapter_path"]), "base_model":parent_manifest["base_model"], "tokenizer_path":parent_manifest.get("tokenizer_path"), "adapter_path":str(output_dir.relative_to(PROJECT_ROOT)), "mode":mode, "policy_version":f"grpo:{mode}:seed{SEED}", "readiness_audit":audit, "logs":logs, "seed":SEED, "profile":TRAINING_PROFILE}
    manifest_path = output_dir / "grpo_manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status":"artifact_saved", "adapter_path":str(output_dir), "manifest_path":str(manifest_path)}


def _reload_grpo_adapter(parent_manifest, artifact, mode):
    reloaded_manifest = dict(parent_manifest, adapter_path=str(artifact["adapter_path"]))
    return _load_sft_policy(reloaded_manifest, mode)


### 9.3 GRPO 训练与 Smoke 调度

In [ ]:
def _write_grpo_jsonl(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in rows) + ("\n" if rows else ""), encoding="utf-8")


def _read_grpo_jsonl(path: Path) -> list[dict[str, object]]:
    if not path.exists(): raise FileNotFoundError(f"阶段产物不存在：{path}")
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _grpo_preflight(stage: str) -> dict[str, object]:
    if not torch.cuda.is_available(): return {"status":"blocked_no_cuda", "stage":stage}
    free, total = torch.cuda.mem_get_info()
    result = {"status":"ready", "stage":stage, "device":torch.cuda.get_device_name(0), "free_memory_mib":free/2**20, "total_memory_mib":total/2**20, "micro_batch_size":GRPO_MICRO_BATCH_SIZE, "gradient_accumulation_steps":GRPO_GRAD_ACCUMULATION_STEPS}
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    (RESULT_DIR / f"{stage}_preflight.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result


def run_grpo_training(group_members, sft_manifest_path=SFT_MANIFEST_PATH, mode="nf4_qlora"):
    if not RUN_GRPO_TRAINING: return {"status":"skipped_by_default"}
    audit = audit_group_members(group_members)
    if audit.get("status") != "ok": return {"status":"invalid_rollout_input", "audit":audit, "optimizer_steps":0}
    if audit.get("mixed_ratio", 0) == 0:
        return {"status":"blocked_no_mixed_groups", "audit":audit, "optimizer_steps":0, "reason":"No genuine group variance; no model update or fabricated reward"}
    sft = read_sft_manifest(sft_manifest_path)
    grouped = defaultdict(list)
    for row in group_members: grouped[(row["prompt_id"],row["task_type"])].append(row)
    ordered = [row for key in sorted(grouped) for row in sorted(grouped[key], key=lambda r:r["group_index"])]
    advantage = build_group_reward_tensors(ordered)["advantages"].reshape(-1)
    # Group Advantage still uses all G=8; only the forward pass is split.
    bf16, tokenizer = _load_sft_policy(sft, "bf16_lora")
    baseline_alignment = audit_policy_logprobs(bf16, tokenizer, ordered, "bf16_lora")
    del bf16
    import gc
    gc.collect(); torch.cuda.empty_cache()
    if baseline_alignment["status"] != "passed": return baseline_alignment
    model, tokenizer = _load_sft_policy(sft, mode)
    alignment = audit_policy_logprobs(model, tokenizer, ordered, mode)
    if alignment["status"] != "passed": return alignment
    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    device = str(model.get_input_embeddings().weight.device)
    chunks = [ordered[start:start+GRPO_MICRO_BATCH_SIZE] for start in range(0,len(ordered),GRPO_MICRO_BATCH_SIZE)]
    batches = [prepare_grpo_batch(chunk,int(pad),device) for chunk in chunks]
    advantages = list(advantage.split(GRPO_MICRO_BATCH_SIZE))
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE)
    logs = causal_lm_grpo_update(model, optimizer, batches, advantages,
                               micro_batch_size=GRPO_MICRO_BATCH_SIZE, gradient_accumulation_steps=GRPO_GRAD_ACCUMULATION_STEPS)
    if logs.get("status") != "completed": return logs
    artifact = save_grpo_adapter(model, tokenizer, sft_manifest_path, mode, audit, logs, parent_manifest=sft)
    reload_check = actual_adapter_reload(model, Path(artifact["adapter_path"]), ordered[0]["prompt_token_ids"])
    metadata = json.loads(Path(artifact["manifest_path"]).read_text())
    metadata.update({"status":"completed", "adapter_reload":reload_check})
    _write_stage_json(Path(artifact["manifest_path"]),metadata)
    return {**artifact,"status":"completed","optimizer_steps":logs["optimizer_steps"],"logs":logs,"readiness_audit":audit,"alignment":alignment,"adapter_reload":reload_check}


In [ ]:
def select_grpo_smoke_prompts(per_task: int = SMOKE_PROMPTS_PER_TASK) -> list[dict[str, object]]:
    if not TRAIN_PATH.exists(): raise FileNotFoundError("缺少 Train 划分")
    frame = __import__("pandas").read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)
    selected = []
    for task, offset in (("equation_numeric", 1), ("bit_manipulation", 2)):
        pool = frame[frame["task_type"] == task]
        for row in pool.sample(n=min(per_task, len(pool)), random_state=SEED + offset).itertuples(index=False):
            selected.append({"id":str(row.id), "task_type":str(row.task_type), "prompt":str(row.prompt), "answer":str(row.answer), "expected_answer":str(row.answer), "source_split":"train"})
    return selected


def run_grpo_rollout_phase():
    if not RUN_GRPO_ROLLOUT: return {"status":"skipped_by_default"}
    def operation():
        sft=read_sft_manifest(); prompts=select_grpo_smoke_prompts()
        rows=run_vllm_group_rollout(prompts,sft["base_model"]["path"],sft["policy_version"],resolve_manifest_path(sft["adapter_path"]))
        return _persist_rollout(rows,GRPO_ROLLOUT_PATH,GRPO_ROLLOUT_MANIFEST_PATH,sft,len(prompts)*GROUP_SIZE)
    return _gpu_stage("grpo_rollout",operation)


def run_grpo_training_phase(mode="nf4_qlora"):
    if not RUN_GRPO_TRAINING: return {"status":"skipped_by_default"}
    def operation():
        rows=_read_rollout(GRPO_ROLLOUT_PATH,GRPO_ROLLOUT_MANIFEST_PATH)
        verified=attach_verifier_rewards(rows,lambda r:independent_answer_verifier(r["task_type"],r["completion"],r["expected_answer"],r["truncated"]))
        _write_grpo_jsonl(RESULT_DIR / "grpo_verified_rollouts.jsonl", verified)
        audit=audit_group_members(verified)
        _write_stage_json(RESULT_DIR / "real_readiness_audit.json",audit)
        return run_grpo_training(verified,SFT_MANIFEST_PATH,mode)
    return _gpu_stage("grpo_training",operation)


def run_grpo_smoke_pipeline(mode="nf4_qlora"):
    if RUN_GRPO_ROLLOUT and RUN_GRPO_TRAINING:
        return {"status":"blocked_kernel_switch_required", "reason":"Run Rollout and Training in separate profiles/processes"}
    if RUN_GRPO_ROLLOUT: return run_grpo_rollout_phase()
    if RUN_GRPO_TRAINING: return run_grpo_training_phase(mode)
    return {"status":"skipped_by_default"}


### 9.4 GPU 开关与 KL 诊断

In [ ]:
print({"causal_lm_update": "implemented", "run_grpo_rollout": RUN_GRPO_ROLLOUT, "run_grpo_training": RUN_GRPO_TRAINING, "reference_kl": "not_implemented; policy_logprob_change only", "KL_COEF": KL_COEF, "micro_batch_size": GRPO_MICRO_BATCH_SIZE, "gradient_accumulation_steps": GRPO_GRAD_ACCUMULATION_STEPS})


## 10. CPU 负样本与 GRPO 输入 Smoke Test

下面的测试覆盖输入缺字段、行为 Logprob 长度错误、非法 Reward、非完整 Group、
错误答案和 Bit 非法输出。它们验证的是工程边界，不是 Qwen3 的能力分数。CPU
Fixture 只制造三种审计类别，正式 Readiness 结果必须来自真实 SFT Checkpoint 的
Group Rollout。


In [ ]:
# Build realistic-shaped records for alignment tests. They are explicitly fixtures.
def _fixture_rollout(prompt_id: str, completion: str, expected: str, task_type: str = "equation_numeric") -> dict[str, object]:
    prompt_ids = [101, 102, 103]
    completion_ids = [11, 12, 13]
    answer = independent_answer_verifier(task_type, completion, expected)
    return {
        "prompt_id": prompt_id,
        "task_type": task_type,
        "prompt": "fixture prompt",
        "expected_answer": expected,
        "prompt_token_ids": prompt_ids,
        "completion": completion,
        "completion_token_ids": completion_ids,
        "response_mask": [0] * len(prompt_ids) + [1] * len(completion_ids),
        "behavior_logprobs": [-0.2, -0.3, -0.4],
        "finish_reason": "stop",
        "truncated": False,
        "policy_version": "cpu_fixture",
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "source": "cpu_fixture_only",
        "fixture": True,
        **answer,
    }

positive = _fixture_rollout("fixture-p", r"reasoning \boxed{42}", "42")
negative = _fixture_rollout("fixture-p", r"reasoning \boxed{41}", "42")
bit_positive = _fixture_rollout("fixture-bit", "Final answer: 10101010", "10101010", "bit_manipulation")
bit_negative = _fixture_rollout("fixture-bit", "Final answer: 1010101", "10101010", "bit_manipulation")
assert positive["correct"] and positive["reward"] == 1.0
assert not negative["correct"] and negative["reward"] == 0.0
assert bit_positive["correct"] and not bit_negative["correct"]
assert _extract_final_answer("Intermediate bits include 10101010; still checking.", "bit_manipulation")[0] is None

records = [positive, negative]
assert audit_group_members(records, group_size=2)["status"] == "ok"
reward_batch = build_group_reward_tensors(records, group_size=2)
assert reward_batch["rewards"].shape == (1, 2)
assert torch.allclose(reward_batch["advantages"], torch.tensor([[1.0, -1.0]]), atol=1e-5)
prepared = prepare_grpo_batch(records, pad_token_id=0)
assert prepared["input_ids"].shape == (2, 6)
assert prepared["response_mask"][:, :3].sum() == 0
assert prepared["response_mask"][:, 3:].sum() == 6
assert prepared["old_logprobs"].shape == (2, 5)

# Negative input tests must fail closed.
missing_reward = dict(positive)
missing_reward.pop("reward")
assert audit_group_members([missing_reward], group_size=1)["status"] == "invalid_input"
bad_logprob = dict(positive, behavior_logprobs=[-0.2])
assert audit_group_members([bad_logprob], group_size=1)["status"] == "invalid_input"
bad_reward = dict(positive, reward=2.0)
assert audit_group_members([bad_reward], group_size=1)["status"] == "invalid_input"
assert audit_group_members([positive], group_size=2)["status"] == "invalid_input"
truncated_positive = _fixture_rollout("fixture-truncated", r"\\boxed{42}", "42")
truncated_positive["truncated"] = True
truncated_result = independent_answer_verifier("equation_numeric", truncated_positive["completion"], "42", truncated=True)
assert not truncated_result["correct"] and truncated_result["verification_method"] == "truncated_output"

process_bonus = reward_process_bonus(torch.tensor([True]), torch.tensor([False]), alpha=0.1)
assert torch.allclose(process_bonus, torch.tensor([0.1]))

# The same objective also runs through the CausalLM-shaped entry point.
tiny_causal = TinyPolicy(vocab_size=128)
tiny_optimizer = torch.optim.AdamW(tiny_causal.parameters(), lr=1e-2)
causal_logs = causal_lm_grpo_update(
    tiny_causal, tiny_optimizer, prepared, reward_batch["advantages"].reshape(-1)
)
assert causal_logs["status"] == "completed"
assert all(math.isfinite(float(value)) for key, value in causal_logs.items() if key != "status")

RESULT_DIR.mkdir(parents=True, exist_ok=True)
final_audit = audit_group_members(cpu_fixture_members)
final_report = {
    "status": "passed",
    "readiness_audit": final_audit,
    "negative_verifier_tests": {
        "numeric_wrong_answer_rejected": True,
        "bit_invalid_answer_rejected": True,
        "missing_reward_rejected": True,
        "logprob_alignment_rejected": True,
        "out_of_range_reward_rejected": True,
        "invalid_group_size_rejected": True,
    },
    "alignment": {
        "prompt_mask_zero": True,
        "response_mask_one": True,
        "old_logprob_shape_aligned": True,
        "verifier_version": OFFICIAL_VERIFIER_VERSION,
    },
    "causal_lm_update_finite": True,
    "gpu_rollout": "not_run",
    "gpu_training": "not_run",
    "fixture_only": True,
}
(RESULT_DIR / "cpu_readiness_audit.json").write_text(json.dumps(final_audit, ensure_ascii=False, indent=2), encoding="utf-8")
(RESULT_DIR / "cpu_grpo_smoke_report.json").write_text(json.dumps(final_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(final_report, ensure_ascii=False, indent=2))


## 本节新增交付说明

- `cpu_readiness_audit.json` 和 `cpu_grpo_smoke_report.json` 只记录 CPU Fixture 与负样本边界测试，明确标记 `fixture_only=true`。
- 真实 GPU 路径仍默认关闭。运行前必须使用最佳 SFT Checkpoint，在 Equation Numeric 和 Bit Manipulation 上生成 `G=8` 候选，并将每条候选交给独立 Verifier。
- 如果真实 Group 全部为 All-correct 或 All-wrong，报告 Zero-Variance Ratio 并回到采样、SFT 或 Drill 诊断；不修改 Reward 制造 Mixed Group。


## 11. SFT Adapter 交接与 Reward 诊断

GRPO 使用同一 SFT Adapter。当前 `policy_logprob_change` 只是行为策略与更新策略的 Logprob 变化，不是 Reference Policy KL；因此 `KL_COEF=0`，不得把它解释为 Reference KL 约束。


In [ ]:
if not (RUN_GRPO_ROLLOUT or RUN_GRPO_TRAINING):
    assert RUN_GRPO is False
assert KL_COEF == 0.0
assert "reference_kl" not in globals()
assert "policy_logprob_change" in globals()
print(json.dumps({"status":"passed", "group_size":GROUP_SIZE, "smoke_prompts_per_task":SMOKE_PROMPTS_PER_TASK, "reference_kl":"not_implemented", "policy_logprob_change_coefficient":KL_COEF, "sft_manifest":str(SFT_MANIFEST_PATH.relative_to(PROJECT_ROOT)), "gpu":"not_run"}, ensure_ascii=False, indent=2))


### GPU 阶段：资源、失败与状态记录
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def _write_stage_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def _gpu_stage(stage: str, operation) -> dict:
    """Preflight + device memory sampler + truthful failure/status report."""
    import gc, threading, subprocess, traceback, importlib.metadata
    started = time.perf_counter()
    stop, samples = threading.Event(), []
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    report_path = RESULT_DIR / f"{stage}_status.json"
    if not torch.cuda.is_available():
        result = {"status": "blocked_no_cuda", "stage": stage}
        _write_stage_json(report_path, result)
        return result
    free, total = torch.cuda.mem_get_info()
    preflight = {"device": torch.cuda.get_device_name(0), "total_mib": total / 2**20,
                 "free_mib": free / 2**20, "torch_cuda": torch.version.cuda,
                 "bf16_supported": torch.cuda.is_bf16_supported()}
    preflight["versions"] = {}
    for package in ("torch", "transformers", "peft", "bitsandbytes", "vllm"):
        try: preflight["versions"][package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: pass
    _write_stage_json(RESULT_DIR / f"{stage}_preflight.json", preflight)
    def monitor():
        while not stop.is_set():
            try:
                output = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True, timeout=3)
                samples.append(float(output.splitlines()[0]))
            except (OSError, ValueError, subprocess.SubprocessError): pass
            stop.wait(0.5)
    thread = threading.Thread(target=monitor, daemon=True)
    torch.cuda.reset_peak_memory_stats()
    thread.start()
    try:
        if "training" in stage and "vllm" in __import__("sys").modules:
            result = {"status": "blocked_kernel_switch_required", "reason": "先结束 vLLM 进程并切换训练 Kernel"}
        elif free / total < 0.80:
            result = {"status": "blocked_gpu_occupied", "reason": "GPU 可用显存不足 80%；不终止其他任务"}
        else:
            result = operation()
            if not isinstance(result, dict) or "status" not in result:
                raise ValueError("阶段必须显式返回 status")
    except Exception as exc:
        message = traceback.format_exc()
        result = {"status": "oom" if isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower() else "failed",
                  "error": repr(exc), "traceback": message}
        (RESULT_DIR / f"{stage}_error.log").write_text(message, encoding="utf-8")
    finally:
        stop.set(); thread.join(timeout=4)
        gc.collect(); torch.cuda.empty_cache()
    result.update({"stage": stage, "wall_time_seconds": time.perf_counter() - started,
                   "preflight": preflight, "peak_memory_allocated_mib": torch.cuda.max_memory_allocated() / 2**20,
                   "peak_device_memory_mib": max(samples) if samples else None})
    _write_stage_json(report_path, result)
    return result


### 阶段交接：真实 Token、原始 Logprob 与 Adapter 指纹
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def _file_sha(path: Path) -> str:
    import hashlib
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _sft_fingerprint(manifest: dict) -> dict:
    adapter = Path(resolve_manifest_path(manifest["adapter_path"]))
    return {"base_model_id": manifest["base_model"]["id"], "policy_version": manifest["policy_version"],
            "manifest_sha256": _file_sha(SFT_MANIFEST_PATH),
            "adapter_files": {p.name: _file_sha(p) for p in sorted(adapter.glob("adapter_*")) if p.is_file()}}


def _chosen_logprob_values(generated) -> list[float]:
    """vLLM 0.11 chosen-token dictionary lookup; never substitute another token."""
    raw, ids = generated.logprobs, list(generated.token_ids or [])
    if raw is None or not ids or len(raw) != len(ids):
        raise ValueError("Missing or misaligned vLLM token logprobs")
    result = []
    for token, candidates in zip(ids, raw):
        if not isinstance(candidates, dict) or token not in candidates:
            raise ValueError(f"Chosen token {token} missing from vLLM Logprob dictionary")
        value = float(candidates[token].logprob)
        if not math.isfinite(value): raise ValueError("non-finite token logprob")
        result.append(value)
    return result


def _persist_rollout(records: list[dict], path: Path, manifest_path: Path, sft: dict, expected_count: int) -> dict:
    if len(records) != expected_count: raise ValueError("Rollout count differs from Smoke budget")
    fingerprint = _sft_fingerprint(sft)
    _check_train_origin(records)
    for row in records:
        if row.get("fixture"): raise ValueError("CPU fixture cannot enter GPU pipeline")
        ids, completion = row["prompt_token_ids"], row["completion_token_ids"]
        if not ids or not completion or len(completion) != len(row["behavior_logprobs"]):
            raise ValueError("Invalid token alignment")
        if row["response_mask"] != [0] * len(ids) + [1] * len(completion):
            raise ValueError("Response Mask differs from stored Token IDs")
        row.update({"source_split": "train", "rollout_quantization": "bf16", "logprob_semantics": "vllm_v1_raw_logprobs",
                    "sampling_distribution": "raw_softmax_T1_topP1_topK_disabled", "sft_fingerprint": fingerprint})
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in records), encoding="utf-8")
    payload = {"schema_version": "nemotron-rollout/v2", "status": "completed", "records": len(records),
               "rollout_path": str(path.relative_to(PROJECT_ROOT)), "rollout_sha256": _file_sha(path),
               "sft_fingerprint": fingerprint, "source_split": "train", "temperature": 1.0, "top_p": 1.0,
               "top_k": -1, "logprobs_mode": "raw_logprobs", "quantization": "bf16",
               "max_new_tokens": MAX_NEW_TOKENS, "max_model_len": MAX_MODEL_LEN}
    _write_stage_json(manifest_path, payload)
    return payload


def _read_rollout(path: Path, manifest_path: Path) -> list[dict]:
    saved = json.loads(manifest_path.read_text())
    if saved.get("status") != "completed" or saved.get("rollout_sha256") != _file_sha(path):
        raise ValueError("Incomplete or modified Rollout artifact")
    if saved.get("sft_fingerprint") != _sft_fingerprint(read_sft_manifest()):
        raise ValueError("SFT Manifest/Adapter changed since Rollout; refuse policy-version mismatch")
    records = [json.loads(s) for s in path.read_text().splitlines() if s.strip()]
    if len(records) != saved["records"]: raise ValueError("Rollout record count mismatch")
    _check_train_origin(records)
    expected_fingerprint = saved["sft_fingerprint"]
    seen = set()
    for row in records:
        key = (row["sample_id"], row["group_index"])
        if key in seen: raise ValueError("Duplicate candidate index in Rollout group")
        seen.add(key)
        if row.get("sft_fingerprint") != expected_fingerprint or row.get("policy_version") != expected_fingerprint["policy_version"]:
            raise ValueError("Per-record SFT policy version mismatch")
        ids, completion = row["prompt_token_ids"], row["completion_token_ids"]
        if not ids or not completion or len(completion) != len(row["behavior_logprobs"]):
            raise ValueError("Invalid stored token/logprob alignment")
        if row["response_mask"] != [0]*len(ids)+[1]*len(completion):
            raise ValueError("Stored Response Mask mismatch")
        if not all(math.isfinite(float(x)) and float(x)<=1e-6 for x in row["behavior_logprobs"]):
            raise ValueError("Invalid sampled-token log probability")
        if row.get("source_split") != "train" or row.get("fixture"):
            raise ValueError("Only real Train Rollout may train")
        if row.get("temperature") != 1.0 or row.get("top_p") != 1.0:
            raise ValueError("This Smoke requires T=1, top_p=1 for raw/behavior logprob equivalence")
    return records


### 策略对齐：BF16 Rollout 与训练侧量化差异
Rollout 与 Training 使用独立进程和 Kernel；持久化完成后退出当前进程，再切换官方环境。


In [ ]:
def audit_policy_logprobs(model, tokenizer, records: list[dict], mode: str) -> dict:
    """Compare actual generated Token IDs without re-tokenizing the completion.

    One real sequence per task is teacher-forced. NF4 is explicitly a different
    target policy from the BF16 rollout; old denominator logprobs stay unchanged.
    """
    chosen, seen = [], set()
    for row in records:
        if row["task_type"] not in seen:
            chosen.append(row); seen.add(row["task_type"])
    deltas, checks = [], []
    model.eval()
    device = model.get_input_embeddings().weight.device
    for row in chosen:
        prompt_ids, response_ids = row["prompt_token_ids"], row["completion_token_ids"]
        rendered_ids = tokenizer.apply_chat_template([{"role":"user", "content":row["prompt"]}], tokenize=True, add_generation_prompt=True, enable_thinking=True)
        # Transformers 4 returns list[int]; Transformers 5 defaults to BatchEncoding.
        if hasattr(rendered_ids, "keys"):
            rendered_ids = rendered_ids["input_ids"]
        if list(rendered_ids) != prompt_ids: raise ValueError("Stored vLLM Prompt Token IDs differ from official Chat Template")
        if row["response_mask"] != [0] * len(prompt_ids) + [1] * len(response_ids):
            raise ValueError("Prompt/Response mask mismatch")
        ids = torch.tensor([prompt_ids + response_ids], device=device)
        with torch.no_grad():
            output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)
            logits = output.logits[0]
            start = len(prompt_ids) - 1
            values = []
            for offset in range(0, len(response_ids), 64):
                count = min(64, len(response_ids) - offset)
                block = logits[start + offset:start + offset + count].float()
                labels = ids[0, start + offset + 1:start + offset + count + 1]
                selected = block.gather(-1, labels[:, None]).squeeze(-1) - torch.logsumexp(block, dim=-1)
                values.extend(selected.cpu().tolist())
        delta = torch.tensor(values) - torch.tensor(row["behavior_logprobs"], dtype=torch.float32)
        deltas.extend(delta.tolist())
        checks.append({"sample_id": row.get("sample_id", row.get("prompt_id")), "task_type": row["task_type"],
                       "response_tokens": len(response_ids), "prompt_tokens": len(prompt_ids),
                       "first_response_action_index": start, "token_mask_alignment": True,
                       "vllm_first_raw_logprob": row["behavior_logprobs"][0], "training_first_raw_logprob": values[0]})
        del output, logits
    absolute = torch.tensor(deltas).abs()
    if not absolute.numel(): raise ValueError("No logprob evidence available")
    # Thresholds are explicit engineering gates, not equality claims or estimated KL.
    mean_limit, p99_limit = ((0.10, 0.60) if mode == "bf16_lora" else (0.50, 3.0))
    result = {"status": "passed" if absolute.mean() <= mean_limit and absolute.quantile(0.99) <= p99_limit else "blocked_policy_distribution_mismatch",
              "training_mode": mode, "rollout_mode": "bf16", "scope": "one_actual_rollout_per_task",
              "tokens_compared": len(deltas), "mean_abs_logprob_delta": float(absolute.mean()),
              "p99_abs_logprob_delta": float(absolute.quantile(0.99)), "max_abs_logprob_delta": float(absolute.max()),
              "signed_mean_logprob_delta": float(torch.tensor(deltas).mean()), "mean_limit": mean_limit,
              "p99_limit": p99_limit, "quantization_difference_expected": mode == "nf4_qlora", "checks": checks}
    _write_stage_json(RESULT_DIR / f"alignment_{mode}.json", result)
    return result


### vLLM 0.11 原始概率与采样语义
显式关闭 top-k/top-p 裁剪、惩罚和模型 generation_config 默认值，使用 T=1。V1 raw_logprobs 才与行为采样分布相同；不从 top-1 代替缺失的采样 Token。


In [ ]:
def _generate_rollout_records(prompt_rows: list[dict], model_source: str, policy_version: str, adapter_path: str, n: int) -> list[dict]:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    tokenizer = AutoTokenizer.from_pretrained(model_source, local_files_only=Path(model_source).exists())
    engine = None
    try:
        engine = LLM(model=model_source, tokenizer=model_source, dtype="bfloat16", enable_lora=True,
                     tensor_parallel_size=1, gpu_memory_utilization=float(os.environ.get("VLLM_GPU_MEMORY_UTILIZATION", "0.80")),
                     max_model_len=MAX_MODEL_LEN, max_num_seqs=8, enforce_eager=True, generation_config="vllm",
                     logprobs_mode="raw_logprobs", seed=SEED)
        sampling = SamplingParams(temperature=1.0, top_p=1.0, top_k=-1, repetition_penalty=1.0,
                                  presence_penalty=0.0, frequency_penalty=0.0, n=n, max_tokens=MAX_NEW_TOKENS,
                                  logprobs=1, seed=SEED, skip_special_tokens=False)
        rendered = [tokenizer.apply_chat_template([{"role":"user","content":r["prompt"]}], tokenize=False, add_generation_prompt=True, enable_thinking=True) for r in prompt_rows]
        if any(len(tokenizer.encode(s, add_special_tokens=False)) + MAX_NEW_TOKENS > MAX_MODEL_LEN for s in rendered):
            raise ValueError(f"Rollout context budget exceeds max_model_len={MAX_MODEL_LEN}")
        outputs = engine.generate(rendered, sampling, lora_request=LoRARequest("sft", 1, adapter_path), use_tqdm=True)
        if len(outputs) != len(prompt_rows): raise ValueError("Rollout request count mismatch")
        rows = []
        for original, request in zip(prompt_rows, outputs):
            if len(request.outputs) != n: raise ValueError("Rollout group size mismatch")
            prompt_ids = list(request.prompt_token_ids or [])
            for index, out in enumerate(request.outputs):
                ids = list(out.token_ids or [])
                rows.append({"sample_id":str(original["id"]), "prompt_id":str(original["id"]),
                    "task_type":original["task_type"], "prompt":original["prompt"], "expected_answer":str(original["answer"]),
                    "completion":out.text, "prompt_token_ids":prompt_ids, "completion_token_ids":ids,
                    "response_mask":[0]*len(prompt_ids)+[1]*len(ids), "behavior_logprobs":_chosen_logprob_values(out),
                    "finish_reason":out.finish_reason, "stop_reason":out.stop_reason, "truncated":out.finish_reason=="length",
                    "policy_version":policy_version, "temperature":1.0, "top_p":1.0, "top_k":-1,
                    "group_index":index, "source":"vllm_rollout", "fixture":False, "trace_verified":False})
        return rows
    finally:
        if engine is not None: del engine
        import gc
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


### Adapter 磁盘重载检查
更新后先保存，再从磁盘加载为独立 PEFT Adapter，比较真实模型 logits；不把目录存在等同于重载成功。


In [ ]:
def actual_adapter_reload(model, adapter_dir: Path, input_ids: list[int]) -> dict:
    """Load saved PEFT weights from disk under a second adapter and compare logits."""
    model.eval()
    original = model.active_adapter
    if isinstance(original, list): original = original[0]
    device = model.get_input_embeddings().weight.device
    probe = torch.tensor([input_ids[:32]], device=device)
    with torch.no_grad():
        before = model(input_ids=probe, attention_mask=torch.ones_like(probe), use_cache=False).logits.float().cpu()
    model.load_adapter(str(adapter_dir), adapter_name="disk_reload_check", is_trainable=False)
    model.set_adapter("disk_reload_check")
    with torch.no_grad():
        after = model(input_ids=probe, attention_mask=torch.ones_like(probe), use_cache=False).logits.float().cpu()
    maximum = float((before - after).abs().max())
    model.set_adapter(original)
    if maximum > 1e-5: raise ValueError(f"Adapter reload logit mismatch: {maximum}")
    return {"status":"passed", "method":"PEFT load_adapter from saved disk weights, compare real-model logits", "max_abs_logit_difference":maximum}


### CPU 机制测试：完整批次与梯度累积等价
同一固定 Group Advantage，在一次 Optimizer Step 内按有效 Response Token 权重累积；不改写 Group Reward。


In [ ]:
# G=8 group advantages stay fixed when only the execution micro-batch changes.
import copy
full_policy = TinyPolicy(vocab_size=128)
micro_policy = copy.deepcopy(full_policy)
large_batch = {key:value.repeat(4,1) for key,value in prepared.items()}
fixed_advantages = reward_batch["advantages"].reshape(-1).repeat(4)
full_optim = torch.optim.SGD(full_policy.parameters(),lr=0.01)
micro_optim = torch.optim.SGD(micro_policy.parameters(),lr=0.01)
full_result = causal_lm_grpo_update(full_policy,full_optim,large_batch,fixed_advantages,micro_batch_size=8,gradient_accumulation_steps=1)
micro_result = causal_lm_grpo_update(micro_policy,micro_optim,large_batch,fixed_advantages,micro_batch_size=1,gradient_accumulation_steps=8)
assert full_result["optimizer_steps"] == micro_result["optimizer_steps"] == 1
maximum = max(float((a-b).abs().max()) for a,b in zip(full_policy.parameters(),micro_policy.parameters()))
assert maximum < 1e-6
_write_stage_json(RESULT_DIR / "cpu_microbatch_equivalence.json",{"status":"passed","fixture_only":True,"members":8,"max_parameter_difference":maximum,"optimizer_steps":1,"micro_batches":8})


### 来源检查：Train 原题和候选索引
持久化与重新读取时核对原始 Train ID、Prompt、答案和任务；不读取 Holdout。


In [ ]:
def _check_train_origin(records):
    import pandas as pd
    frame = pd.read_csv(TRAIN_PATH,dtype=str,keep_default_na=False)
    if set(frame["split"]) != {"train"}: raise ValueError("Train split metadata mismatch")
    lookup = frame.set_index("id").to_dict("index")
    for row in records:
        original = lookup.get(str(row["sample_id"]))
        if original is None or row["prompt"] != original["prompt"] or str(row["expected_answer"]) != original["answer"] or row["task_type"] != original["task_type"]:
            raise ValueError("Rollout is not an unchanged official Train instance")


## GPU Smoke 独立执行入口：GRPO
Rollout Kernel 只设置 `RUN_GRPO_ROLLOUT=1`；退出进程确认显存释放后，在训练 Kernel 只设置 `RUN_GRPO_TRAINING=1`。两次执行使用相同 `GPU_SMOKE_RUN_ID`、数据目录和 Notebook 03 的 SFT Manifest。


In [ ]:
if RUN_GRPO_ROLLOUT and RUN_GRPO_TRAINING:
    grpo_gpu_status = {"status":"blocked_kernel_switch_required", "reason":"分别在 vLLM 和训练环境执行；两阶段开关不能同时开启"}
elif RUN_GRPO_ROLLOUT:
    grpo_gpu_status = run_grpo_rollout_phase()
elif RUN_GRPO_TRAINING:
    grpo_gpu_status = run_grpo_training_phase(os.environ.get("RL_QUANTIZATION_MODE", "nf4_qlora"))
else:
    grpo_gpu_status = {"status":"skipped_by_default"}
print(json.dumps(grpo_gpu_status,ensure_ascii=False,indent=2))
